In [8]:
import numpy as np
import cv2
import os

In [9]:
highway_path = r"C:\Users\mkrol\studia\semestr6\zaawansowane_algorytmy_wizyjne\lab02\highway"
highway_ROI = cv2.imread(os.path.join(highway_path, "ROI.bmp"), cv2.IMREAD_GRAYSCALE)

highway_input = []
highway_groundtruth = []

for i in range(470, 1701):
    input_name = f"in{i:06d}.jpg"
    gt_name = f"gt{i:06d}.png"
    
    img = cv2.imread(os.path.join(highway_path, "input", input_name), cv2.IMREAD_GRAYSCALE)
    gt = cv2.imread(os.path.join(highway_path, "groundtruth", gt_name), cv2.IMREAD_GRAYSCALE)
    
    if img is not None and gt is not None:
        highway_input.append(img)
        highway_groundtruth.append(gt)

print(f"Wczytano {len(highway_input)} klatek.")

Wczytano 1231 klatek.


In [10]:
office_path = r"C:\Users\mkrol\studia\semestr6\zaawansowane_algorytmy_wizyjne\lab02\office"
office_ROI = cv2.imread(os.path.join(office_path, "ROI.bmp"), cv2.IMREAD_GRAYSCALE)

office_input = []
office_groundtruth = []

for i in range(570, 2051):
    input_name = f"in{i:06d}.jpg"
    gt_name = f"gt{i:06d}.png"
    
    img = cv2.imread(os.path.join(office_path, "input", input_name), cv2.IMREAD_GRAYSCALE)
    gt = cv2.imread(os.path.join(office_path, "groundtruth", gt_name), cv2.IMREAD_GRAYSCALE)
    
    if img is not None and gt is not None:
        office_input.append(img)
        office_groundtruth.append(gt)

print(f"Wczytano {len(office_input)} klatek.")

Wczytano 1481 klatek.


In [11]:
pedestrian_path = r"C:\Users\mkrol\studia\semestr6\zaawansowane_algorytmy_wizyjne\lab02\pedestrian"
pedestrian_ROI = cv2.imread(os.path.join(pedestrian_path, "ROI.bmp"), cv2.IMREAD_GRAYSCALE)

pedestrian_input = []
pedestrian_groundtruth = []

for i in range(300, 1100):
    input_name = f"in{i:06d}.jpg"
    gt_name = f"gt{i:06d}.png"
    
    img = cv2.imread(os.path.join(pedestrian_path, "input", input_name), cv2.IMREAD_GRAYSCALE)
    gt = cv2.imread(os.path.join(pedestrian_path, "groundtruth", gt_name), cv2.IMREAD_GRAYSCALE)
    
    if img is not None and gt is not None:
        pedestrian_input.append(img)
        pedestrian_groundtruth.append(gt)

print(f"Wczytano {len(pedestrian_input)} klatek.")

Wczytano 800 klatek.


In [21]:
total_TP = 0
total_FP = 0
total_FN = 0

for i in range(1, len(highway_input)):
    IG_f = highway_input[i].astype('float32')
    IG_s = highway_input[i-1].astype('float32')
    diff = np.abs(IG_f - IG_s).astype('uint8')

    _, binary_mask = cv2.threshold(diff, 20, 255, cv2.THRESH_BINARY)

    binary_mask = cv2.medianBlur(binary_mask, 5)
    binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_OPEN, np.ones((3,3)))
    binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_CLOSE, np.ones((3,3), np.uint8))

    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary_mask)
    I_VIS = cv2.cvtColor(highway_input[i], cv2.COLOR_GRAY2BGR) 
    
    if stats.shape[0] > 1: 
        pi = np.argmax(stats[1:, 4]) + 1 
        x, y, w, h, area = stats[pi]
        cv2.rectangle(I_VIS, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(I_VIS, f"Area: {int(area)}", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

    gt = highway_groundtruth[i]
    
    total_TP += np.sum((binary_mask == 255) & (gt == 255))
    total_FP += np.sum((binary_mask == 255) & (gt == 0))
    total_FN += np.sum((binary_mask == 0) & (gt == 255))
    
    cv2.imshow('Detekcja Ruchu', I_VIS)
    cv2.imshow("", binary_mask)
    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

Precision = total_TP / (total_TP + total_FP) if (total_TP + total_FP) > 0 else 0
Recall = total_TP / (total_TP + total_FN) if (total_TP + total_FN) > 0 else 0
F1 = (2 * Precision * Recall) / (Precision + Recall) if (Precision + Recall) > 0 else 0


print(f"Precision: {Precision:.4f}")
print(f"Recall:    {Recall:.4f}")
print(f"F1-Score:  {F1:.4f}")        

cv2.destroyAllWindows()

Precision: 0.9508
Recall:    0.4068
F1-Score:  0.5698


In [16]:
total_TP = 0
total_FP = 0
total_FN = 0

for i in range(1, len(office_input)):
    IG_f = office_input[i].astype('float32')
    IG_s = office_input[i-1].astype('float32')
    diff = np.abs(IG_f - IG_s).astype('uint8')

    _, binary_mask = cv2.threshold(diff, 20, 255, cv2.THRESH_BINARY)

    binary_mask = cv2.GaussianBlur(binary_mask, (3,3), 0)
    binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_OPEN, np.ones((3,3)))
    binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_CLOSE, np.ones((7,7), np.uint8))

    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary_mask)
    I_VIS = cv2.cvtColor(office_input[i], cv2.COLOR_GRAY2BGR) 
    
    if stats.shape[0] > 1: 
        pi = np.argmax(stats[1:, 4]) + 1 
        x, y, w, h, area = stats[pi]
        cv2.rectangle(I_VIS, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(I_VIS, f"Area: {int(area)}", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

    gt = office_groundtruth[i]
    
    total_TP += np.sum((binary_mask == 255) & (gt == 255))
    total_FP += np.sum((binary_mask == 255) & (gt == 0))
    total_FN += np.sum((binary_mask == 0) & (gt == 255))
    
    cv2.imshow('Detekcja Ruchu', I_VIS)
    
    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

Precision = total_TP / (total_TP + total_FP) if (total_TP + total_FP) > 0 else 0
Recall = total_TP / (total_TP + total_FN) if (total_TP + total_FN) > 0 else 0
F1 = (2 * Precision * Recall) / (Precision + Recall) if (Precision + Recall) > 0 else 0


print(f"Precision: {Precision:.4f}")
print(f"Recall:    {Recall:.4f}")
print(f"F1-Score:  {F1:.4f}")        

cv2.destroyAllWindows()

Precision: 0.8701
Recall:    0.0018
F1-Score:  0.0036


In [17]:
total_TP = 0
total_FP = 0
total_FN = 0

for i in range(1, len(pedestrian_input)):
    IG_f = pedestrian_input[i].astype('float32')
    IG_s = pedestrian_input[i-1].astype('float32')
    diff = np.abs(IG_f - IG_s).astype('uint8')

    _, binary_mask = cv2.threshold(diff, 20, 255, cv2.THRESH_BINARY)

    binary_mask = cv2.GaussianBlur(binary_mask, (5,5), 0)
    binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_OPEN, np.ones((3,3)))
    binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_CLOSE, np.ones((5,5), np.uint8))

    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary_mask)
    I_VIS = cv2.cvtColor(pedestrian_input[i], cv2.COLOR_GRAY2BGR) 
    
    if stats.shape[0] > 1: 
        pi = np.argmax(stats[1:, 4]) + 1 
        x, y, w, h, area = stats[pi]
        cv2.rectangle(I_VIS, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(I_VIS, f"Area: {int(area)}", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

    gt = pedestrian_groundtruth[i]
    
    total_TP += np.sum((binary_mask == 255) & (gt == 255))
    total_FP += np.sum((binary_mask == 255) & (gt == 0))
    total_FN += np.sum((binary_mask == 0) & (gt == 255))
    
    cv2.imshow('Detekcja Ruchu', I_VIS)
    
    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

Precision = total_TP / (total_TP + total_FP) if (total_TP + total_FP) > 0 else 0
Recall = total_TP / (total_TP + total_FN) if (total_TP + total_FN) > 0 else 0
F1 = (2 * Precision * Recall) / (Precision + Recall) if (Precision + Recall) > 0 else 0


print(f"Precision: {Precision:.4f}")
print(f"Recall:    {Recall:.4f}")
print(f"F1-Score:  {F1:.4f}")        

cv2.destroyAllWindows()

Precision: 0.6944
Recall:    0.1228
F1-Score:  0.2088
